In [1]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  path = "/content/drive/MyDrive/COMP720Project/"
except Exception:
  path = "../data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pip
pip.main(['install', 'pandas', 'numpy', 'tqdm'])

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


Requirement already satisfied: pandas in /usr/local/lib/python3.12/dist-packages (2.2.2)

Requirement already satisfied: numpy in /usr/local/lib/python3.12/dist-packages (1.26.4)

Requirement already satisfied: tqdm in /usr/local/lib/python3.12/dist-packages (4.67.3)

Requirement already satisfied: python-dateutil>=2.8.2 in /usr/local/lib/python3.12/dist-packages (from pandas) (2.9.0.post0)

Requirement already satisfied: pytz>=2020.1 in /usr/local/lib/python3.12/dist-packages (from pandas) (2025.2)

Requirement already satisfied: tzdata>=2022.7 in /usr/local/lib/python3.12/dist-packages (from pandas) (2026.3)

Requirement already satisfied: six>=1.5 in /usr/local/lib/python3.12/dist-packages (from python-dateutil>=2.8.2->pandas) (1.17.0)

0

In [3]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

NumExpr defaulting to 12 threads.

In [4]:
!pip install replay-rec[torch] -q
!pip install pyspark==3.3.2

### Load shared encodings (produced by `encoding.ipynb`)

In [5]:
from replay.preprocessing import LabelEncoder

ENCODED_DIR = Path(path) / "encoded"
ENCODER_PATH = ENCODED_DIR / "encoder"
INTERACTIONS_PATH = ENCODED_DIR / "encoded_interactions.parquet"
ITEM_FEATURES_PATH = ENCODED_DIR / "item_features_encoded.parquet"

# consumed later by ContentAwareEmbedder / FeaturesReader -- same file, kept under its
# original name so the rest of the notebook doesn't need to change
PATH_ENCODED_FEATURES = ITEM_FEATURES_PATH

encoder = LabelEncoder.load(ENCODER_PATH)
encoded_interactions = pd.read_parquet(INTERACTIONS_PATH)
item_features_encoded = pd.read_parquet(ITEM_FEATURES_PATH)
item_features_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15711 entries, 0 to 15710
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   item_id            15711 non-null  int64 
 1   title              15711 non-null  object
 2   genres             15711 non-null  object
 3   genre_names        15711 non-null  object
 4   item_numerics      15711 non-null  object
 5   embeddings         15711 non-null  object
 6   keyword_embedding  15711 non-null  object
dtypes: int64(1), object(6)
memory usage: 859.3+ KB


### Content-Based preprocessing: keep only liked interactions, attach item features

In [6]:
# from replay.preprocessing.filters import LowRatingFilter

# # Drop low satisfaction interactions, keep only interactions with rating >= 3
# rating_filter = LowRatingFilter(value=3, rating_column="rating")
# encoded_interactions = rating_filter.transform(encoded_interactions)

# # SasRec (implicit) doesn't need the raw rating value # TODO: May be use rating
# encoded_interactions = encoded_interactions.drop(columns=["rating"])

In [7]:
# merge genre + numeric item features into interactions BEFORE baking
# (same as your other item-level joins)
encoded_interactions = encoded_interactions.merge(
    item_features_encoded[["item_id", "genres", "item_numerics"]],
    on="item_id", how="inner")

### Train/val/test split

In [8]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

In [9]:
from replay.data.nn.utils import groupby_sequences


def bake_data(full_data):
    grouped_interactions = groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")
    return grouped_interactions


train_events = bake_data(train_events)

validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)

test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events

,user_id,rating,timestamp,item_id,genres,item_numerics
0,0,"[4.0, 1.0, 4.0, 2.0, 1.0, 5.0, 5.0, 1.0, 4.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[346, 297, 1979, 7826, 1300, 42, 1562, 552, 26...","[[4, 7, 9], [7], [1, 4, 18], [7], [4, 7, 10], ...","[[0.018571429, 0.0078200735, 0.1931624, 0.8], ..."
1,1,"[4.0, 1.0, 5.0, 3.0, 3.0, 5.0, 1.0, 5.0, 5.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[208, 507, 9096, 303, 2576, 599, 1482, 3061, 2...","[[9, 1, 5], [17, 5, 4], [1, 17], [1, 5, 9], [1...","[[0.05, 0.14069435, 0.21538462, 0.72307694], [..."
2,2,"[3.0, 1.0, 4.0, 3.5, 4.0, 3.5, 4.0, 4.0, 3.5, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[146, 2516, 662, 1996, 256, 595, 944, 399, 340...","[[2, 4, 15], [1, 4, 18], [7], [1, 2, 7, 14], [...","[[0.057142857, 0.08363617, 0.2034188, 0.730769..."
3,3,"[3.0, 2.0, 3.0, 4.0, 5.0, 3.0, 2.0, 3.0, 2.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1088, 2355, 3984, 3850, 50, 4007, 1992, 603, ...","[[2, 1, 15], [1, 5, 7, 17], [2, 7, 1, 10], [18...","[[0.046214286, 0.19588153, 0.22564103, 0.67692..."
4,4,"[4.0, 3.0, 3.0, 1.0, 5.0, 4.0, 4.0, 3.0, 3.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[208, 402, 414, 507, 9096, 2576, 912, 1482, 59...","[[9, 1, 5], [7, 10], [2, 7, 19], [17, 5, 4], [...","[[0.05, 0.14069435, 0.21538462, 0.72307694], [..."
...,...,...,...,...,...,...
200942,200942,"[4.0, 3.5, 5.0, 5.0, 3.5, 5.0, 4.0, 5.0, 3.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[4, 1087, 216, 435, 52, 82, 1088, 13156, 12285...","[[2, 1, 15], [2, 1, 15], [7, 5], [1, 15], [2, ...","[[0.015714286, 0.26521066, 0.20683761, 0.63076..."
200943,200943,"[5.0, 2.5, 4.0, 5.0, 5.0, 4.0, 3.5, 4.0, 4.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[216, 541, 539, 11476, 7835, 212, 155, 9121, 1...","[[7, 5], [11, 4], [13, 17, 7], [7, 4], [1, 15,...","[[0.035714287, 0.009693679, 0.24273504, 0.7615..."
200944,200944,"[5.0, 4.0, 4.0, 5.0, 4.0, 4.0, 4.0, 4.0, 3.0, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[532, 1706, 492, 2515, 434, 511, 1199, 717, 32...","[[2, 1, 17], [7, 14], [1, 2, 7], [1, 17], [1, ...","[[0.08571429, 0.12046151, 0.22222222, 0.769230..."
200945,200945,"[4.0, 5.0, 4.0, 3.5, 5.0, 4.5, 3.5, 4.0, 2.5, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[216, 308, 435, 82, 467, 7835, 80, 5139, 6560,...","[[7, 5], [7, 10, 18], [1, 15], [2, 9, 1], [4, ...","[[0.035714287, 0.009693679, 0.24273504, 0.7615..."


In [10]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [11]:
for df in (train_events, validation_events, test_events):  # and test_df / predict_df if applicable
    df["rating"] = df["rating"].apply(lambda seq: np.asarray(seq, dtype=np.float32))

In [12]:
data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
TEST_PATH = data_dir / "test.parquet"

In [13]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(TEST_PATH)

## Training

### Tensor Schema

In [14]:
import torch
from replay.data.nn import TensorMap, TensorSchema, TensorFeatureInfo, TensorFeatureSource
from replay.data import FeatureHint, FeatureType, FeatureSource
from replay.nn.embedding import SequenceEmbedding
from replay.nn.sequential.twotower import FeaturesReader

class ContentAwareEmbedder(torch.nn.Module):
    def __init__(self, train_schema, full_schema, content_feature_configs, embedding_dim):
        """
        content_feature_configs: list of dicts, e.g.
            [{"name": "embeddings", "path": DESC_EMB_PATH},
             {"name": "keyword_embedding", "path": KEYWORD_EMB_PATH}]
        """
        super().__init__()
        self.base_embedder = SequenceEmbedding(schema=train_schema)   # handles item_id, genres, etc.

        self.content_tables = torch.nn.ModuleDict()
        self.projections = torch.nn.ModuleDict()
        self.norms = torch.nn.ModuleDict()

        for cfg in content_feature_configs:
            name = cfg["name"]
            reader = FeaturesReader(schema=full_schema, metadata={name: {}}, path=cfg["path"])
            self.register_buffer(f"content_table_{name}", reader[name])
            self.projections[name] = torch.nn.Linear(reader[name].size(-1), embedding_dim)
            self.norms[name] = torch.nn.LayerNorm(embedding_dim)

        self._feature_names = [cfg["name"] for cfg in content_feature_configs]

    def reset_parameters(self):
        self.base_embedder.reset_parameters()
        for name in self._feature_names:
            torch.nn.init.xavier_uniform_(self.projections[name].weight)

    def forward(self, feature_tensors, feature_names=None):
        embeddings = self.base_embedder(feature_tensors, feature_names)   # item_id + genres, etc.
        item_ids = feature_tensors["item_id"]

        for name in self._feature_names:
            table = getattr(self, f"content_table_{name}")
            vecs = table[item_ids]
            embeddings[name] = self.norms[name](self.projections[name](vecs))

        return embeddings

    def get_item_weights(self, indices=None):
        return self.base_embedder.get_item_weights(indices)

In [15]:
# Full schema: item_id + embeddings -- used ONLY to construct FeaturesReader
NUM_UNIQUE_ITEMS = item_features_encoded["item_id"].nunique()
NUM_GENRES = int(item_features_encoded["genres"].explode().dropna().astype(int).max()) + 1
EMBEDDING_DIM = 128
full_schema = TensorSchema([
    TensorFeatureInfo(
        name="item_id", is_seq=True, feature_type=FeatureType.CATEGORICAL,
        cardinality=NUM_UNIQUE_ITEMS, padding_value=NUM_UNIQUE_ITEMS,
        embedding_dim=EMBEDDING_DIM, feature_hint=FeatureHint.ITEM_ID,
        feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")],
    ),
    TensorFeatureInfo(
        name="embeddings", is_seq=False, feature_type=FeatureType.NUMERICAL_LIST,
        tensor_dim=1024, embedding_dim=EMBEDDING_DIM,
        feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "embeddings")],
    ),
    # TODO: Add this
    TensorFeatureInfo(
    name="keyword_embedding", is_seq=True, feature_type=FeatureType.NUMERICAL_LIST,
    tensor_dim=128, embedding_dim=EMBEDDING_DIM,
    feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "keyword_embedding")],
    ),
])

# Training schema: item_id ONLY -- this is what actually matches your train/val parquet columns
train_schema = TensorSchema([
    full_schema["item_id"],
    TensorFeatureInfo(
            name="rating",
            is_seq=True,
            feature_type=FeatureType.NUMERICAL,
            tensor_dim=1,
            embedding_dim=EMBEDDING_DIM,  # must match item_id's, SumAggregator requires it
            feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, "rating")],
    ),
    TensorFeatureInfo(
        name="genres", is_seq=True, feature_type=FeatureType.CATEGORICAL_LIST,
        cardinality=NUM_GENRES, padding_value=NUM_GENRES, embedding_dim=EMBEDDING_DIM,
        feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "genres")],
    ),
        TensorFeatureInfo(
        name="item_numerics", is_seq=True, feature_type=FeatureType.NUMERICAL_LIST,
        tensor_dim=4, embedding_dim=EMBEDDING_DIM,
        feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "item_numerics")],
    ),
])

In [16]:
from replay.nn.transform.template import make_default_sasrec_transforms
from replay.data.nn import ParquetModule

BATCH_SIZE = 128
MAX_SEQ_LEN = 60
MAX_GENRES_PER_ITEM = item_features_encoded["genres"].apply(len).max()

transforms = make_default_sasrec_transforms(train_schema)   # only expects item_id -- matches your lean parquet

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": train_schema["item_id"].padding_value},
        "rating": {"shape": MAX_SEQ_LEN + 1, "padding": 0.0},
        "genres": {"shape": [MAX_SEQ_LEN + 1, MAX_GENRES_PER_ITEM], "padding": NUM_GENRES},
        "item_numerics": {"shape": [MAX_SEQ_LEN + 1, 4], "padding": 0.0},
        },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": train_schema["item_id"].padding_value},
        "rating": {"shape": MAX_SEQ_LEN, "padding": 0.0},
        "genres": {"shape": [MAX_SEQ_LEN, MAX_GENRES_PER_ITEM], "padding": NUM_GENRES},
        "item_numerics": {"shape": [MAX_SEQ_LEN, 4], "padding": 0.0},
        "ground_truth": {"shape": 1, "padding": -1},
    },
}

parquet_module = ParquetModule(
    train_path=TRAIN_PATH, validate_path=VAL_PATH,
    batch_size=BATCH_SIZE, metadata=train_metadata, transforms=transforms,
)

/tmp/ipykernel_4117/1733474489.py:26: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [17]:
from replay.nn.agg import SumAggregator
from replay.nn.mask import DefaultAttentionMask
from replay.nn.loss import CE
from replay.nn.sequential import PositionAwareAggregator, SasRecTransformerLayer, SasRecBody, SasRec

DROPOUT = 0.2
NUM_HEADS = 4
NUM_BLOCKS = 2


body = SasRecBody(
    embedder=ContentAwareEmbedder(
        train_schema=train_schema,
        full_schema=full_schema,
        content_feature_configs=[
            {'name': 'embeddings','path':PATH_ENCODED_FEATURES},
            {"name": "keyword_embedding", "path": PATH_ENCODED_FEATURES},
            ],
        embedding_dim=EMBEDDING_DIM,
    ),
    embedding_aggregator=PositionAwareAggregator(
        embedding_aggregator=SumAggregator(embedding_dim=EMBEDDING_DIM),
        max_sequence_length=MAX_SEQ_LEN,
        dropout=DROPOUT,
    ),
    attn_mask_builder=DefaultAttentionMask(
        reference_feature_name=train_schema.item_id_feature_name,
        num_heads=NUM_HEADS,
    ),
    encoder=SasRecTransformerLayer(
        embedding_dim=EMBEDDING_DIM,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        dropout=DROPOUT,
        activation="relu",
        hidden_dim=EMBEDDING_DIM * 4,
    ),
    output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
)
sasrec = SasRec(
    body=body,
    loss=CE(ignore_index=train_schema[train_schema.item_id_feature_name].padding_value),
)

In [18]:
from replay.nn.lightning import LightningModule

model = LightningModule(sasrec)

In [ ]:
# type(train_metadata["train"]["genres"]["shape"])
# assert isinstance(train_metadata["train"]["genres"]["shape"], list)
# assert isinstance(train_metadata["validate"]["genres"]["shape"], list)

In [19]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
import lightning as L

from replay.nn.lightning.callback import ComputeMetricsCallback

from lightning.pytorch.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor="recall@10", mode="max", patience=5, verbose=True)


checkpoint_callback = ModelCheckpoint(
    dirpath="sasrec/checkpoints",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall", 'precision'], # TODO: Use precision
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
    # verbose=False,
)

csv_logger = CSVLogger(save_dir="sasrec/logs/train", name="SasRec-example")

trainer = L.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback, validation_metrics_callback, early_stop],
    logger=csv_logger,
    accelerator="gpu",
    gradient_clip_val=1.0,
)

trainer.fit(model, datamodule=parquet_module)

INFO: GPU available: True (cuda), used: True


GPU available: True (cuda), used: True

INFO: TPU available: False, using: 0 TPU cores


TPU available: False, using: 0 TPU cores

INFO: You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

Metric recall@10 improved by 0.008 >= min_delta = 0.0. New best score: 0.169

INFO: Epoch 4, global step 7850: 'recall@10' reached 0.16909 (best 0.16909), saving model to '/content/sasrec/checkpoints/epoch=4-step=7850.ckpt' as top 1


Epoch 4, global step 7850: 'recall@10' reached 0.16909 (best 0.16909), saving model to '/content/sasrec/checkpoints/epoch=4-step=7850.ckpt' as top 1

INFO: `Trainer.fit` stopped: `max_epochs=5` reached.


`Trainer.fit` stopped: `max_epochs=5` reached.

In [20]:
best_model_path = checkpoint_callback.best_model_path
best_score = checkpoint_callback.best_model_score
print(best_model_path, best_score)

/content/sasrec/checkpoints/epoch=4-step=7850.ckpt tensor(0.1691, device='cuda:0')


### Inference

To obtain model scores, we will load the weights from the best checkpoint. To do this, we use the LightningModule, providing the path to the checkpoint and the model instance.

In [21]:
sasrec = SasRec(
    body=body,
    loss=CE(ignore_index=train_schema[train_schema.item_id_feature_name].padding_value),
)

best_model = LightningModule.load_from_checkpoint(best_model_path, model=sasrec)
best_model.eval()

LightningModule(
  (model): SasRec(
    (body): SasRecBody(
      (embedder): ContentAwareEmbedder(
        (base_embedder): SequenceEmbedding(
          (feature_embedders): ModuleDict(
            (item_id): CategoricalEmbedding(
              (emb): Embedding(15704, 128, padding_idx=15703)
            )
            (rating): NumericalEmbedding(
              (linear): Linear(in_features=1, out_features=128, bias=True)
            )
            (genres): CategoricalEmbedding(
              (emb): EmbeddingBag(21, 128, mode='sum', padding_idx=20)
            )
            (item_numerics): NumericalEmbedding(
              (linear): Linear(in_features=4, out_features=128, bias=True)
            )
          )
        )
        (content_tables): ModuleDict()
        (projections): ModuleDict(
          (embeddings): Linear(in_features=1024, out_features=128, bias=True)
          (keyword_embedding): Linear(in_features=128, out_features=128, bias=True)
        )
        (norms): ModuleDic

In [24]:
inference_metadata = {
    "predict": {
        "user_id": {},   # empty dict -- genuine scalar, not a list
        "rating": {"shape": MAX_SEQ_LEN, "padding": 0.0},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": train_schema["item_id"].padding_value},
        "genres": {"shape": [MAX_SEQ_LEN, MAX_GENRES_PER_ITEM], "padding": NUM_GENRES},
        "item_numerics": {"shape": [MAX_SEQ_LEN, 4], "padding": 0.0},
    }
}

parquet_module = ParquetModule(
    predict_path=TEST_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

/tmp/ipykernel_4117/1636007793.py:11: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


In [25]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="sasrec/logs/test", name="SasRec-example")

TOPK = [1, 5, 10, 20]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(callbacks=[pandas_prediction_callback], logger=csv_logger, inference_mode=True)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

INFO: GPU available: True (cuda), used: True


GPU available: True (cuda), used: True

INFO: TPU available: False, using: 0 TPU cores


TPU available: False, using: 0 TPU cores

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

Predicting: |          | 0/? [00:00<?, ?it/s]

### Calculating metrics

test_gt is already encoded, so we can use it for computing metrics.

In [26]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall, NDCG
from replay.metrics.torch_metrics_builder import metrics_to_df

In [27]:
result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK), NDCG(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

In [28]:
metrics_to_df(result_metrics)

k,1,5,10,20
MAP,0.026962,0.049652,0.057470,0.063005
NDCG,0.026962,0.060764,0.079877,0.100262
Precision,0.026962,0.018966,0.015431,0.011769
Recall,0.026962,0.094831,0.154309,0.235380


In [29]:
pandas_res

,user_id,item_id,score
0,0,3824,5.882158
0,0,3197,5.83087
0,0,7746,5.82167
0,0,6188,5.767824
0,0,301,5.571974
...,...,...,...
200946,200946,10126,4.068416
200946,200946,6540,3.872832
200946,200946,6542,3.866052
200946,200946,7309,3.826937


In [30]:
pandas_res.to_csv("temp/sasrec_result.csv")

In [31]:
# !rm -rf drive/MyDrive/COMP720Project/hbf
!mkdir drive/MyDrive/COMP720Project/hbf
!cp -r sasrec drive/MyDrive/COMP720Project/hbf/sasrec
!cp -r temp drive/MyDrive/COMP720Project/chf/temp

cp: cannot create directory 'drive/MyDrive/COMP720Project/chf/temp': No such file or directory


### Explanation

In [46]:
import numpy as np

def cosine_sim(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

def explain_recommendation(user_history_item_ids, recommended_item_id, movie_df,
                            desc_weight=0.5, genre_weight=0.2, keyword_weight=0.3):
    """
    movie_df indexed by item_id (encoded ids), with columns:
      'embeddings'        -- raw bge-m3 vector (pre-projection)
      'keyword_embedding' -- raw SVD-reduced TF-IDF vector (pre-projection)
      'genres'            -- set/list of genre ids (or names, whichever you prefer for display)
      'title'
    """
    candidate = movie_df.loc[recommended_item_id]
    candidate_desc = candidate["embeddings"]
    candidate_kw = candidate["keyword_embedding"]
    candidate_genres = set(candidate["genre_names"])

    scored = []
    for hist_id in user_history_item_ids[-20:]:   # cap history length, most recent 20
        hist = movie_df.loc[hist_id]
        desc_sim = cosine_sim(candidate_desc, hist["embeddings"])
        keyword_sim = cosine_sim(candidate_kw, hist["keyword_embedding"])
        hist_genres = set(hist["genre_names"])
        overlap = candidate_genres & hist_genres
        genre_sim = len(overlap) / max(len(candidate_genres | hist_genres), 1)

        combined = desc_weight * desc_sim + genre_weight * genre_sim + keyword_weight * keyword_sim
        scored.append({
            "item_id": hist_id, "title": hist["title"],
            "desc_sim": desc_sim, "keyword_sim": keyword_sim,
            "genre_overlap": overlap, "combined": combined,
        })

    scored.sort(key=lambda x: x["combined"], reverse=True)
    return scored[0], scored

In [41]:
def format_explanation(best_match, recommended_title):
    reasons = []
    if best_match["genre_overlap"]:
        reasons.append(f"shares the {', '.join(sorted(best_match['genre_overlap']))} genre(s)")
    if best_match["desc_sim"] > 0.6:
        reasons.append("has a similar theme/plot")
    if best_match["keyword_sim"] > 0.5:
        reasons.append("touches on similar story elements")

    if not reasons:
        reasons.append("is broadly similar in style")

    return f"Because you watched \"{best_match['title']}\", which {' and '.join(reasons)}, we think you'll like \"{recommended_title}\"."

In [42]:
from tqdm import tqdm
def add_explanations(pandas_res, user_histories, movie_df):
    explanations = []
    for _, row in tqdm(pandas_res.iterrows()):
        user_id = row["user_id"]
        rec_item_id = row["item_id"]   # still encoded id at this point
        history = user_histories.get(user_id, [])
        if not history:
            explanations.append(None)
            continue
        best_match, _ = explain_recommendation(history, rec_item_id, movie_df)
        rec_title = movie_df.loc[rec_item_id, "title"]
        explanations.append(format_explanation(best_match, rec_title))

    pandas_res = pandas_res.copy()
    pandas_res["explanation"] = explanations
    return pandas_res

In [35]:
user_histories = test_events.set_index("user_id")["item_id"].to_dict()

In [36]:
MAX_SAMPLE = 1000

In [38]:
# Rebuild movie_df indexed by (encoded) item_id for the explanation helpers below.
# item_features_encoded already carries title/genres/embeddings/keyword_embedding.
movie_df = item_features_encoded.set_index("item_id", drop=False)
movie_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15711 entries, 645 to 7171
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   item_id            15711 non-null  int64 
 1   title              15711 non-null  object
 2   genres             15711 non-null  object
 3   genre_names        15711 non-null  object
 4   item_numerics      15711 non-null  object
 5   embeddings         15711 non-null  object
 6   keyword_embedding  15711 non-null  object
dtypes: int64(1), object(6)
memory usage: 981.9+ KB


In [47]:
pandas_res_with_explanations = add_explanations(pandas_res[:MAX_SAMPLE], user_histories, movie_df)
pandas_res_with_explanations[["user_id", "item_id", "score", "explanation"]].head(10)


0it [00:00, ?it/s]
30it [00:00, 298.90it/s]
60it [00:00, 299.35it/s]
90it [00:00, 296.74it/s]
120it [00:00, 297.08it/s]
150it [00:00, 297.60it/s]
181it [00:00, 299.40it/s]
211it [00:00, 297.38it/s]
241it [00:00, 297.56it/s]
272it [00:00, 300.09it/s]
303it [00:01, 300.24it/s]
334it [00:01, 301.52it/s]
365it [00:01, 301.61it/s]
396it [00:01, 300.82it/s]
427it [00:01, 302.76it/s]
458it [00:01, 302.62it/s]
489it [00:01, 298.07it/s]
519it [00:01, 298.20it/s]
549it [00:01, 297.82it/s]
580it [00:01, 298.75it/s]
611it [00:02, 300.62it/s]
642it [00:02, 301.50it/s]
673it [00:02, 301.69it/s]
704it [00:02, 300.43it/s]
735it [00:02, 295.83it/s]
765it [00:02, 296.92it/s]
795it [00:02, 296.54it/s]
826it [00:02, 298.31it/s]
857it [00:02, 300.16it/s]
888it [00:02, 299.77it/s]
918it [00:03, 299.67it/s]
949it [00:03, 299.87it/s]
1000it [00:03, 299.13it/s]


,user_id,item_id,score,explanation
0,0,3824,5.882158,"Because you watched ""Secrets & Lies"", which sh..."
0,0,3197,5.83087,"Because you watched ""Cat on a Hot Tin Roof"", w..."
0,0,7746,5.82167,"Because you watched ""American History X"", whic..."
0,0,6188,5.767824,"Because you watched ""EverAfter"", which shares ..."
0,0,301,5.571974,"Because you watched ""Jude"", which shares the D..."
0,0,10785,5.481284,"Because you watched ""Secrets & Lies"", which sh..."
0,0,2136,5.436394,"Because you watched ""Shine"", which shares the ..."
0,0,7703,5.334821,"Because you watched ""A Room with a View"", whic..."
0,0,3150,5.251916,"Because you watched ""Shine"", which shares the ..."
0,0,11235,5.213207,"Because you watched ""Cat on a Hot Tin Roof"", w..."


In [48]:
list(pandas_res_with_explanations['explanation'])[0]

'Because you watched "Secrets & Lies", which shares the Drama genre(s) and has a similar theme/plot and touches on similar story elements, we think you\'ll like "Secrets & Lies".'

In [49]:
pandas_res_with_explanations.to_csv("drive/MyDrive/COMP720Project/hbf/explanation_res.csv")